# Abstract Base Class - training

BaseModel is an abstract class. Every model (LSTM, GRU, TFT, etc.) will subclass it and only needs to implement preprocess() and train(). Everything else (data loading, caching, evaluation and saving) is inherited. This avoids copy-pasting the same logic 6 times.

Data split logic:

+ Main dataset: sales_train_evaluation.csv (1941 days of labels in total)
+ Train: d_1-d_1773
+ Val: d_1774-d_1885 (112 days = 3×28 context + 1×28 prediction target)
+ Test: d_1886–d_1941  (56 days = 1×28 context + 1×28 prediction target)
+ This allows models to compute lag features (up to 28 day lag) using the context window in val/test, without data leakage between the splits.

This notebook breaks down BaseModel step by step for learning purposes. The final class is in base_model.py (outside notebook folder)

In [1]:
# Limit of 3 additional packages
# !pip install numpy pandas lightgbm

In [2]:
import os
import random
import pickle
import json
import torch

from abc import ABC, abstractmethod
import numpy as np
import pandas as pd


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'}")

Device: CPU


# 1. Class skeleton

Abstract methods force subclasses to implement specific methods. Every model must have a model_name(), preprocess() and train().

In [3]:
# Define class skeleton with abstract methods. Document what preprocess and train must do in each subclass

class BaseModel(ABC):

    QUANTILES   = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975]
    PRED_LENGTH = 28
    SEED        = 25

    def __init__(self, data_dir="data", output_dir="outputs"):
        random.seed(self.SEED)
        np.random.seed(self.SEED)
        torch.manual_seed(self.SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        self.device     = DEVICE
        self.data_dir   = data_dir
        self.output_dir = output_dir
        os.makedirs(data_dir,   exist_ok=True)
        os.makedirs(output_dir, exist_ok=True)

        self.train_raw = self.val_raw = self.test_raw = None
        self.calendar  = self.prices  = self.item_weights = None
        self.train_processed = self.val_processed = self.test_processed = None
        self.model = None

    @property
    @abstractmethod
    def model_name(self) -> str: ...

    @abstractmethod
    def preprocess(self): ...
    # Input:  self.train_raw, self.val_raw, self.test_raw
    # Output: self.train_processed, self.val_processed, self.test_processed
    #         (format is model-specific: tensors, lgb.Dataset, DataFrames, etc.)

    @abstractmethod
    def train(self): ...
    # Input:  self.train_processed, self.val_processed
    # Must:
    #   1. Set self.model
    #   2. Save weights → output_dir/{model_name}.pth (or .pkl for LGBM)

    def load_and_split_data(self): pass
    def evaluate(self):           pass
    def run(self):                pass

# Sanity check
try:
    b = BaseModel()
except TypeError as e:
    print(e)

Can't instantiate abstract class BaseModel without an implementation for abstract methods 'model_name', 'preprocess', 'train'


# 2. Data processing

Changes and notes from Luke's data pipeline (2/4):
1. Do not average price per item and store. When merging sales with prices, use the exact daily price of every item.
2. Melt merged dataframe so that each row only has one item and one day
3. Merge calendar.csv also and set the right data types
4. Some items have missing values for sell_price (they were not on sale for the whole time period). In these cases, forward fill the last available price (or 0 if none available). Then create a new column called 'is_available' so that models take into account out of stock / new / retired items.
5. Check that there are no missing values in the merged dataframe
6. Revenue weight for each item should be calculated as follows: multiply **daily** sales * price, then sum the total revenue for each item during the **last 28 training days only** (as per Kaggle requirement), then normalise so that the total item weights sum to 1. 
7. Categorical encoding issue: assigning each categorical id (store_id, state_id etc) an integer implies a fake distance between them, which will cause bias in embeddings and tree models. Skip this in the base class, but handle it in subclasses - see section 3. How to create your own model subclass

In [5]:
# Step 1: load data
data_dir = "data"
cache = os.path.join(data_dir, "raw_split.pkl")

if os.path.exists(cache):
    print("Loading cached data split...")
    with open(cache, "rb") as f:
        d = pickle.load(f)
else:
    print("Downloading M5 data...")
    base     = "https://huggingface.co/datasets/kashif/M5/resolve/main"
    sales    = pd.read_csv(f"{base}/sales_train_evaluation.csv")
    calendar = pd.read_csv(f"{base}/calendar.csv")
    prices   = pd.read_csv(f"{base}/sell_prices.csv")

    print(sales.shape, calendar.shape, prices.shape)
# (30490, 1947)  (1969, 14)  (6841121, 4)

(30490, 1947) (1969, 14) (6841121, 4)


In [6]:
# Step 2: melt sales to long format
id_cols  = [c for c in sales.columns if not c.startswith("d_")]
day_cols = [c for c in sales.columns if c.startswith("d_")]

sales_long = sales[id_cols + day_cols].melt(
    id_vars=id_cols, var_name='d', value_name='sales'
)
sales_long['d_num'] = sales_long['d'].str[2:].astype(int)

print(sales_long.shape)        # (59,132,490, 9)
print(sales_long.head(3))

(59181090, 9)
                              id        item_id    dept_id   cat_id store_id  \
0  HOBBIES_1_001_CA_1_evaluation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
1  HOBBIES_1_002_CA_1_evaluation  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1   
2  HOBBIES_1_003_CA_1_evaluation  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1   

  state_id    d  sales  d_num  
0       CA  d_1      0      1  
1       CA  d_1      0      1  
2       CA  d_1      0      1  


In [7]:
# Step 3: merge calendar data and set dtypes
for col in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
    calendar[col] = calendar[col].fillna('none').astype('category')
calendar['weekday'] = calendar['weekday'].astype('category')
calendar['date']    = pd.to_datetime(calendar['date'])
for col in ['wday', 'month', 'year', 'snap_CA', 'snap_TX', 'snap_WI']:
    calendar[col] = calendar[col].astype('int8')

sales_long = sales_long.merge(calendar, on='d', how='left')

print(sales_long.dtypes)
display(sales_long.sample(5))

id                         str
item_id                    str
dept_id                    str
cat_id                     str
store_id                   str
state_id                   str
d                          str
sales                    int64
d_num                    int64
date            datetime64[us]
wm_yr_wk                 int64
weekday               category
wday                      int8
month                     int8
year                      int8
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap_CA                   int8
snap_TX                   int8
snap_WI                   int8
dtype: object


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,d_num,date,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
54212483,HOUSEHOLD_2_167_CA_1_evaluation,HOUSEHOLD_2_167,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1779,0,1779,2015-12-12,...,1,12,-33,none,none,none,none,0,1,1
49067626,HOBBIES_1_072_CA_4_evaluation,HOBBIES_1_072,HOBBIES_1,HOBBIES,CA_4,CA,d_1610,0,1610,2015-06-26,...,7,6,-33,none,none,none,none,0,0,0
30789449,HOUSEHOLD_1_086_WI_2_evaluation,HOUSEHOLD_1_086,HOUSEHOLD_1,HOUSEHOLD,WI_2,WI,d_1010,0,1010,2013-11-03,...,2,11,-35,none,none,none,none,1,1,1
2306110,HOUSEHOLD_1_511_TX_3_evaluation,HOUSEHOLD_1_511,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_76,0,76,2011-04-14,...,6,4,-37,none,none,none,none,0,0,1
2567046,FOODS_3_613_CA_2_evaluation,FOODS_3_613,FOODS_3,FOODS,CA_2,CA,d_85,0,85,2011-04-23,...,1,4,-37,none,none,none,none,0,0,0


In [8]:
# Step 4: merge daily prices
sales_long = sales_long.merge(
    prices[['store_id', 'item_id', 'wm_yr_wk', 'sell_price']],  # prices are actually set weekly
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='left'
)

display(sales_long[['id', 'd', 'wm_yr_wk', 'sell_price']].sample(5))

,id,d,wm_yr_wk,sell_price
58825045,HOUSEHOLD_1_128_CA_4_evaluation,d_1930,11615,2.67
5442659,HOBBIES_1_201_TX_2_evaluation,d_179,11126,5.22
39012671,HOUSEHOLD_1_156_TX_2_evaluation,d_1280,11426,NaN
42237500,FOODS_3_528_CA_3_evaluation,d_1386,11441,3.98
57716816,FOODS_3_071_WI_3_evaluation,d_1893,11610,1.34


In [9]:
# Step 5: add is_available flag and forward-fill missing prices
sales_long['is_available'] = sales_long['sell_price'].notna().astype('int8')

sales_long = sales_long.sort_values(['id', 'd_num']).reset_index(drop=True)

sales_long['sell_price'] = (
    sales_long.groupby('id')['sell_price']
    .transform(lambda x: x.ffill().fillna(0.0))  
)

print(sales_long.groupby('is_available')['sell_price'].describe())


                   count      mean       std   min   25%   50%   75%     max
is_available                                                                
0             12299413.0  0.000000  0.000000  0.00  0.00  0.00  0.00    0.00
1             46881677.0  4.409438  3.406106  0.01  2.18  3.47  5.84  107.32


Notes:
- is_available=0 rows have sell_price=0 (pre-launch) or ffilled price (temporary pause or post-retirement)
- The output above tells us that all unavailable items are pre-launch since they have no last known price

In [10]:
# Step 6: check missing values
missing = sales_long.isnull().sum()
missing = missing[missing > 0]
assert len(missing) == 0, f"Missing values found:\n{missing}"
print("No missing values. Shape:", sales_long.shape)

No missing values. Shape: (59181090, 24)


In [11]:
# Step 7: split into train/val/test
PRED_LENGTH = 28
total_days = len(day_cols)                        # 1941
train_end  = total_days - 6 * PRED_LENGTH         # 1773
val_end    = total_days - 2 * PRED_LENGTH         # 1885

train_raw = sales_long[sales_long['d_num'] <= train_end].reset_index(drop=True)
val_raw   = sales_long[(sales_long['d_num'] > train_end) & (sales_long['d_num'] <= val_end)].reset_index(drop=True)
test_raw  = sales_long[sales_long['d_num'] > val_end].reset_index(drop=True)

assert train_raw['d_num'].nunique() == 1773
assert val_raw['d_num'].nunique()   == 112
assert test_raw['d_num'].nunique()  == 56
print(f"Train: d_1–d_{train_end} | Val: d_{train_end+1}-d_{val_end} | Test: d_{val_end+1}-d_{total_days}")

Train: d_1–d_1773 | Val: d_1774-d_1885 | Test: d_1886-d_1941


In [12]:
# Step 8: calculate revenue weights on last 28 training days only
last28 = train_raw[train_raw['d_num'] > train_end - 28]
train_rev = (last28['sales'] * last28['sell_price']).groupby(last28['id']).sum()
item_weights = (train_rev / train_rev.sum()).rename('weight')

print(f"Weights sum to: {item_weights.sum():.6f}")    # should be 1.0
print(f"Non-zero items: {(item_weights > 0).sum()} / {len(item_weights)}")
print(item_weights.sort_values(ascending=False).head(5))

Weights sum to: 1.000000
Non-zero items: 27040 / 30490
id
HOBBIES_1_354_TX_3_evaluation    0.004617
HOBBIES_1_158_TX_3_evaluation    0.004371
FOODS_3_120_CA_3_evaluation      0.003082
FOODS_1_096_WI_2_evaluation      0.002459
HOBBIES_1_354_TX_2_evaluation    0.002038
Name: weight, dtype: float64


In [13]:
# sense check
print([train_raw.id.nunique(), train_rev.shape, item_weights.shape])

[30490, (30490,), (30490,)]


## 2.1 Full preprocessing function: load_and_split_data

In [ ]:
# Put it all together in one function. Run once and cache to disk. Every subclass reuses the same data split: train 1773 days, val 112 days, test 56 days

def load_and_split_data(self):
    """
    Downloads (or loads from cache) M5 data and processes it into long-format dataframes (1 row per item and day). 
    Split into train (d_1-d_1773), val (d_1774-d_1885), and test (d_1886-d_1941).
    Save raw splits to cache for loading when training models.

    Output: self.train_raw, self.val_raw, self.test_raw, self.item_weights
    """
    # Step 1: load data
    cache = os.path.join(self.data_dir, "raw_split.pkl")

    if os.path.exists(cache):
        with open(cache, "rb") as f:
            d = pickle.load(f)
        print("Loaded cached data splits.")
    else:
        base     = "https://huggingface.co/datasets/kashif/M5/resolve/main"
        sales    = pd.read_csv(f"{base}/sales_train_evaluation.csv")
        calendar = pd.read_csv(f"{base}/calendar.csv")
        prices   = pd.read_csv(f"{base}/sell_prices.csv")
        print("Downloaded M5 data.")

        # Step 2: melt sales to long format
        id_cols  = [c for c in sales.columns if not c.startswith("d_")]
        day_cols = [c for c in sales.columns if c.startswith("d_")]
        sales_long = sales[id_cols + day_cols].melt(
            id_vars=id_cols, var_name='d', value_name='sales'
        )
        sales_long['d_num'] = sales_long['d'].str[2:].astype(int)

        # Step 3: merge calendar data and set dtypes
        for col in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
            calendar[col] = calendar[col].fillna('none').astype('category')
        calendar['weekday'] = calendar['weekday'].astype('category')
        calendar['date']    = pd.to_datetime(calendar['date'])
        for col in ['wday', 'month', 'year', 'snap_CA', 'snap_TX', 'snap_WI']:
            calendar[col] = calendar[col].astype('int8')
        sales_long = sales_long.merge(calendar, on='d', how='left')

        # Step 4: merge daily prices
        sales_long = sales_long.merge(
            prices[['store_id', 'item_id', 'wm_yr_wk', 'sell_price']],
            on=['store_id', 'item_id', 'wm_yr_wk'], how='left'
        )

        # Step 5: add is_available flag and forward-fill missing prices
        sales_long['is_available'] = sales_long['sell_price'].notna().astype('int8')
        sales_long = sales_long.sort_values(['id', 'd_num']).reset_index(drop=True)
        sales_long['sell_price'] = (
            sales_long.groupby('id')['sell_price']
            .transform(lambda x: x.ffill().fillna(0.0))
        )

        # Step 6: check missing values
        missing = sales_long.isnull().sum()
        missing = missing[missing > 0]
        assert len(missing) == 0, f"Missing values found:\n{missing}"

        # Step 7: split into train/val/test
        total_days = len(day_cols)
        train_end  = total_days - 6 * self.PRED_LENGTH   # 1773
        val_end    = total_days - 2 * self.PRED_LENGTH   # 1885

        train_raw = sales_long[sales_long['d_num'] <= train_end].reset_index(drop=True)
        val_raw   = sales_long[(sales_long['d_num'] > train_end) & (sales_long['d_num'] <= val_end)].reset_index(drop=True)
        test_raw  = sales_long[sales_long['d_num'] > val_end].reset_index(drop=True)

        assert train_raw['d_num'].nunique() == 1773
        assert val_raw['d_num'].nunique()   == 112
        assert test_raw['d_num'].nunique()  == 56
        print(f"Train: d_1-d_{train_end} | Val: d_{train_end+1}-d_{val_end} | Test: d_{val_end+1}-d_{total_days}")

        # Step 8: calculate revenue weights on last 28 training days only
        last28 = train_raw[train_raw['d_num'] > train_end - 28]
        train_rev = (last28['sales'] * last28['sell_price']).groupby(last28['id']).sum()
        item_weights = (train_rev / train_rev.sum()).rename('weight')

        # Step 9: save data splits to cache
        d = dict(train_raw=train_raw, val_raw=val_raw, test_raw=test_raw,
                    item_weights=item_weights)

        # save all splits and weights into raw_split.pkl
        with open(cache, "wb") as f:
            pickle.dump(d, f)
        print(f"Cached to {cache}")

    self.train_raw    = d["train_raw"]
    self.val_raw      = d["val_raw"]
    self.test_raw     = d["test_raw"]
    self.item_weights = d["item_weights"]
    print("Finished data processing.")

    return self.train_raw, self.val_raw, self.test_raw, self.item_weights

BaseModel.load_and_split_data = load_and_split_data

In [15]:
# Test the function

class ConcreteModel(BaseModel):
    """Minimal concrete subclass so we can instantiate BaseModel for testing."""
    @property
    def model_name(self): return "test_model"
    def preprocess(self): pass
    def train(self):      pass


def test_load_and_data_split():
    model = ConcreteModel(data_dir="data")
    model.load_and_split_data()

    train, val, test, weights = model.load_and_split_data()

    # Test 1: correct number of unique days in each split
    assert train['d_num'].nunique() == 1773, "Train should have 1773 days"
    assert val['d_num'].nunique()   == 112,  "Val should have 112 days"
    assert test['d_num'].nunique()  == 56,   "Test should have 56 days"
    print("Test 1 passed: split sizes correct")

    # Test 2: no temporal overlap between splits
    assert train['d_num'].max() < val['d_num'].min(), "Train and val overlap"
    assert val['d_num'].max()   < test['d_num'].min(), "Val and test overlap"
    print("Test 2 passed: no temporal overlap")

    # Test 3: no missing values in any split
    for name, df in [("train", train), ("val", val), ("test", test)]:
        assert df.isnull().sum().sum() == 0, f"Missing values in {name}"
    print("Test 3 passed: no missing values")

    # Test 4: is_available is binary and sell_price is non-negative
    for name, df in [("train", train), ("val", val), ("test", test)]:
        assert df['is_available'].isin([0, 1]).all(), f"is_available not binary in {name}"
        assert (df['sell_price'] >= 0).all(), f"Negative sell_price in {name}"
    print("Test 4 passed: is_available binary, sell_price non-negative")

    # Test 5: item_weights sum to 1 and are all non-negative
    assert abs(model.item_weights.sum() - 1.0) < 1e-6, "Weights do not sum to 1"
    assert (model.item_weights >= 0).all(), "Negative weights found"
    print("Test 5 passed: item_weights valid")

    print("\nAll tests passed.")


test_load_and_data_split()

Train: d_1-d_1773 | Val: d_1774-d_1885 | Test: d_1886-d_1941
Cached to data\raw_split.pkl
Loading cached data split...
Test 1 passed: split sizes correct
Test 2 passed: no temporal overlap
Test 3 passed: no missing values
Test 4 passed: is_available binary, sell_price non-negative
Test 5 passed: item_weights valid

All tests passed.


# 3. How to create your own model subclass

Suggestions (Yen): 
1. LSTM, GRU, Hierarchical LSTM should have consistent preprocess methods - check how you are encoding categorical variables and creating embeddings and lags.
2. For LGBM / LGBM+NN, use lgb.Dataset and set categorical_feature = [list of cat_features]
3. For TFT, just use raw string categories in TimeSeriesDataset without encoding

In [17]:
# Make sure you have base_model.py in your working directory. Import the following packages and BaseModel class

import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from base_model import BaseModel

In [18]:
# Define your model class by inheriting from BaseModel. 
# Set model_name and define the methods preprocess and train (feel free to add functions like _make_tennsors below. 
# I've used a very basic linear regression example

class LinearModel(BaseModel):

    model_name = "linear"

    # ------------------------------------------------------------------
    # 1. preprocess
    #    Input:  self.train_raw, self.val_raw, self.test_raw  (long format)
    #    Output: self.train_processed, self.val_processed, self.test_processed
    #            each is a (X, y) tuple of float32 tensors, shape (N_rows, n_features)
    # ------------------------------------------------------------------
    def preprocess(self):
        self.train_processed = self._make_tensors(self.train_raw)
        self.val_processed   = self._make_tensors(self.val_raw)
        self.test_processed  = self._make_tensors(self.test_raw)

    def _make_tensors(self, df: pd.DataFrame):
        """
        Build a flat (X, y) dataset from long-format rows.
        Features: lag-28 sales, sell_price, wday, month, is_available
        Target:   sales
        Rows with no lag-28 value (d_num <= 28) are dropped.
        """
        # merge lag-28 sales onto each row
        lag = (df[['id', 'd_num', 'sales']]
               .assign(d_num=lambda x: x['d_num'] + 28)
               .rename(columns={'sales': 'lag_28'}))
        df = df.merge(lag, on=['id', 'd_num'], how='inner')

        feature_cols = ['lag_28', 'sell_price', 'wday', 'month', 'is_available']
        X = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        y = torch.tensor(df['sales'].values,       dtype=torch.float32).unsqueeze(1)
        return X, y

    # ------------------------------------------------------------------
    # 2. train
    #    Input:  self.train_processed, self.val_processed
    #    Must:   set self.model, save weights to output_dir/linear.pth
    # ------------------------------------------------------------------
    def train(self, epochs=20, lr=1e-3, batch_size=4096):
        X_train, y_train = self.train_processed
        X_val,   y_val   = self.val_processed

        n_features  = X_train.shape[1]
        self.model  = nn.Linear(n_features, 1).to(self.device)
        optimiser   = torch.optim.Adam(self.model.parameters(), lr=lr)
        loss_fn     = nn.MSELoss()

        dataset = torch.utils.data.TensorDataset(X_train, y_train)
        loader  = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

        for epoch in range(epochs):
            self.model.train()
            for X_batch, y_batch in loader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                optimiser.zero_grad()
                loss_fn(self.model(X_batch), y_batch).backward()
                optimiser.step()

            self.model.eval()
            with torch.no_grad():
                val_loss = loss_fn(
                    self.model(X_val.to(self.device)),
                    y_val.to(self.device)
                ).item()
            print(f"Epoch {epoch+1:02d}/{epochs} | val MSE: {val_loss:.4f}")

        torch.save(self.model.state_dict(),
                   f"{self.output_dir}/{self.model_name}.pth")

In [ ]:
# Test your class - make sure preprocess and train run without errors
m = LinearModel()
m.load_and_split_data()  # takes about 4mins
m.preprocess()
# m.train() 

# 4. Evaluation (WIP)

TODO (Yen):

- Define a separate function that takes the raw splits, filters for selected stores and filters for the top num_items by highest total sales in the last num_days (calculated in the training split only). the arguments of the function are: store_ids (list, default: ['CA_3']), num_items (default: 100), last_num_days (default: 365)

- Define a function to standardise the output format of all model predictions like the picture attached. Each row contains an id (same as in train/val/test sets) and predicts the last 28 days (d_1886 to d_1941).

In [ ]:
# # Define evaluate()
# # Compute WSPL

# def evaluate(self) -> dict:
# # Input:  self.test_processed
# # Must:
# #   1. Load weights from output_dir/{model_name}.pth (or .pkl for LGBM)
# #   2. Predict on final 28 days of test_raw (d_1914–d_1941)
# #      using d_1886–d_1913 as context
# #   3. Save CSV → output_dir/{model_name}_preds.csv
# #      Columns: id | day_ahead (1–28) | q0.025 | q0.05 | … | q0.975
# #      Rows:    N_series × 28
# #   4. Compute and print WRMSSE + WSPL against test targets

#     preds_path = os.path.join(self.output_dir, f"{self.model_name}_preds.csv")
#     assert os.path.exists(preds_path), "Run train() first to generate predictions."

#     preds_df  = pd.read_csv(preds_path)
#     test_vals  = self.test_raw[[c for c in self.test_raw.columns
#                                 if c.startswith("d_")]].values.astype(np.float32)   # (N, 56)
#     train_vals = self.train_raw[[c for c in self.train_raw.columns
#                                  if c.startswith("d_")]].values.astype(np.float32)  # (N, 1773)

#     q_cols = [f"q{q}" for q in self.QUANTILES]
#     N      = len(self.train_raw)

#     # Naïve seasonal scale: mean |y_t - y_{t-7}| over training window
#     scale = np.abs(train_vals[:, 7:] - train_vals[:, :-7]).mean(axis=1).clip(min=1e-8)  # (N,)

#     wspl_per_series = np.zeros(N)

#     for i in range(N):
#         series_id = self.train_raw.iloc[i]["id"]
#         row = preds_df[preds_df["id"] == series_id].sort_values("day_ahead")
#         if row.empty:
#             continue

#         pinball = 0.0
#         for q, qcol in zip(self.QUANTILES, q_cols):
#             q_preds = row[qcol].values        # (56,)
#             y_true  = test_vals[i]            # (56,)
#             errors  = y_true - q_preds
#             pinball += np.maximum(q * errors, (q - 1) * errors).mean()

#         wspl_per_series[i] = (pinball / len(self.QUANTILES)) / scale[i]

#     wspl    = float(np.sum(self.item_weights * wspl_per_series))
#     results = {"model": self.model_name, "wspl": wspl}

#     out_path = os.path.join(self.output_dir, f"{self.model_name}_results.json")
#     with open(out_path, "w") as f:
#         json.dump(results, f, indent=2)

#     print(f"WSPL ({self.model_name}): {wspl:.6f}  →  saved to {out_path}")
#     return results

# BaseModel.evaluate = evaluate

In [ ]:
# # Define run() to call the full pipeline

# def run(self):
#     print(f"\n{'='*50}\n  Pipeline: {self.model_name}\n{'='*50}")
#     self.load_and_split_data()
#     self.preprocess()
#     self.train()
#     return self.evaluate()

# BaseModel.run = run